# Exercise 09: Plotly



In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect('../data/checking-logs.sqlite')

query = """
    SELECT uid, timestamp
    FROM checker
    WHERE uid NOT LIKE 'admin%'
      AND uid IS NOT NULL
      AND labname = 'project1'
      AND status = 'ready'
"""
checker = pd.read_sql(query, conn)
conn.close()

# Preparing data

In [2]:
checker['timestamp'] = pd.to_datetime(checker['timestamp'])
checker['date'] = checker['timestamp'].dt.date

# daily → cumulative
daily = checker.groupby(['date', 'uid']).size().reset_index(name='commits')

pivot = daily.pivot(index='date', columns='uid', values='commits').fillna(0)

date_range = pd.date_range(pivot.index.min(), pivot.index.max(), freq='D').date
pivot = pivot.reindex(date_range, fill_value=0)

cumulative = pivot.cumsum()

users = cumulative.columns
dates = cumulative.index


# Plotting graph

In [3]:
fig = go.Figure()

for user in users:
    fig.add_trace(go.Scatter(
        x=[dates[0]], 
        y=[cumulative.iloc[0][user]], 
        mode='lines+markers',
        name=user,
        line=dict(width=3),
        marker=dict(size=8)
    ))

frames = []
for k in range(1, len(dates)):
    frame_data = []
    for user in users:
        frame_data.append(go.Scatter(
            x=dates[:k+1],
            y=cumulative.iloc[:k+1][user]
        ))
    frames.append(go.Frame(data=frame_data, name=str(dates[k])))

fig.frames = frames

fig.update_layout(
    title='Dynamic of commits per user in project1',
    title_font_size=30,
    xaxis=dict(range=[dates[0], dates[-1]], title='Date', title_font_size=15),
    yaxis=dict(range=[0, cumulative.max().max() * 1.1], title='Commits', title_font_size=15),
    height=700,
    width=1200,
    updatemenus=[dict(
        type="buttons",
        buttons=[dict(label="Play",
                      method="animate",
                      args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True)])]
    )]
)

fig.show()